In [1]:
# ==============================================================================
# CELDA 0: PIENZA COMMAND CENTER - BOOTSTRAP (CODESPACES / STREAMLIT READY)
# ==============================================================================
import os
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. AUTENTICACIÓN SOSTENIBLE (Modo Local/Producción)
print("🔐 Autenticando identidad en Google Cloud...")
# NOTA: Para que esto funcione, necesitas tu archivo JSON de Service Account de GCP.
# Reemplaza la ruta con la ubicación real de tu archivo JSON en el Codespace.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/workspaces/pienza/observatory/service-account.json" 
print("✅ Credenciales configuradas en el entorno.")

# 2. CONFIGURACIÓN DEL PROYECTO
PROJECT_ID   = '645009831643'
DATASET_CORE = 'pienza_mini'
DATASET_BIG  = 'pienza_big'

# 3. INICIALIZACIÓN DE CLIENTE (La puerta de entrada a tus datos)
print(f"📡 Conectando al ecosistema BigQuery (Project: {PROJECT_ID})...")
try:
    client = bigquery.Client(project=PROJECT_ID)
    print("✅ Cliente BigQuery conectado con éxito.")
except Exception as e:
    print(f"❌ Error de conexión. ¿Ya subiste tu archivo JSON? Detalle: {e}")

# 4. VISUAL CANON (OPUS LAB THEME) - Preparado para tus gráficas
OPUS_PURPLE = '#440154'
OPUS_TEAL   = '#21918c'
OPUS_GREY   = '#FAFAFA'
OPUS_TEXT   = '#121212'

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.facecolor': OPUS_GREY, 'axes.facecolor': OPUS_GREY,
    'text.color': OPUS_TEXT, 'axes.titlecolor': OPUS_PURPLE,
    'axes.titleweight': 'bold', 'font.family': 'sans-serif'
})

print("\n--- SYSTEM READY: BQ Ecosistema Operativo ---")

🔐 Autenticando identidad en Google Cloud...
✅ Credenciales configuradas en el entorno.
📡 Conectando al ecosistema BigQuery (Project: 645009831643)...
✅ Cliente BigQuery conectado con éxito.

--- SYSTEM READY: BQ Ecosistema Operativo ---


In [2]:
# ==============================================================================
# CELDA 1 (FIXED): DATA INGESTION & TEMPORAL LOCKDOWN (BIGQUERY)
# ==============================================================================
# Purpose: Fetch only the necessary raw data (Week 6 focus) from BigQuery
#          and apply the initial sanity filters (Fixing the SQL NULL Paradox).
# ==============================================================================
print("⏳ Iniciando Extracción desde BigQuery (Enfocado en Semana 6)...")

# 1. THE SQL EXTRACTION
# FIX: 'OR reason_primary_fk IS NULL' rescata los 67 viajes ACCEPTED
query_w6 = f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET_CORE}.v_ML_Supervised`
    WHERE (reason_primary_fk != 7 OR reason_primary_fk IS NULL)
      AND offer_timestamp >= '2025-09-26 00:00:00'
      AND offer_timestamp <= '2025-10-01 23:59:59'
"""

# Ejecutar la consulta y convertir a Pandas
df_raw_w6 = client.query(query_w6).to_dataframe()

# Eliminar columnas duplicadas si el view de BQ las trajo por error
df_raw_w6 = df_raw_w6.loc[:, ~df_raw_w6.columns.duplicated()]

# 2. SANITY CHECK Y REINDEXADO
df_raw_w6 = df_raw_w6.reset_index(drop=True)

print(f"✅ Ingesta completada. Total de registros W6: {len(df_raw_w6)}")
print(f"-> Rango temporal: {df_raw_w6['offer_timestamp'].min()} a {df_raw_w6['offer_timestamp'].max()}")

# 3. PREPARACIÓN DEL TARGET DE VERDAD (Para futura validación)
df_raw_w6['reason_primary_fk'] = df_raw_w6['reason_primary_fk'].fillna(9999)

display(df_raw_w6[['session_fk', 'offer_timestamp', 'upfront_fare', 'reason_primary_fk']].head())

⏳ Iniciando Extracción desde BigQuery (Enfocado en Semana 6)...


✅ Ingesta completada. Total de registros W6: 780
-> Rango temporal: 2025-09-26 05:59:21 a 2025-10-01 09:56:26


,session_fk,offer_timestamp,upfront_fare,reason_primary_fk
0,SID0053,2025-09-26 05:59:40,119.73,1.0
1,SID0053,2025-09-26 06:00:15,82.40,1.0
2,SID0053,2025-09-26 06:00:23,110.33,6.0
3,SID0053,2025-09-26 06:00:54,70.15,1.0
4,SID0053,2025-09-26 06:01:05,69.66,1.0


In [3]:
# ==============================================================================
# CELDA 2: EL COALESCE GEOESPACIAL (LA SINGULARIDAD DEL DÓNDE)
# ==============================================================================
# Purpose: Replicate the creation of 'final_zone_id' ensuring 100% parity
#          with the training pipeline to avoid Training-Serving Skew.
# ==============================================================================
import numpy as np

print("⏳ Ejecutando Master Coalesce Geoespacial en W6...")

# 1. EL MAPA CANÓNICO DE IDs (Inmutable)
id_map = {
    -1: -1, 41: 0, 42: 0, 46: 0, 43: 1, 65: 2, 62: 2, 44: 2, 36: 2, 49: 3, 52: 3,
    35: 3, 50: 4, 58: 4, 25: 5, 31: 5, 63: 6, 39: 6, 51: 7, 33: 7, 37: 8, 53: 8,
    48: 8, 60: 9, 57: 10, 12: 10, 32: 10, 24: 11, 40: 12, 45: 13, 59: 13, 61: 14,
    38: 14, 34: 15, 30: 16, 66: 16, 17: 17, 14: 17, 22: 17, 16: 18, 13: 18, 11: 19,
    15: 20, 21: 21, 20: 21, 19: 21, 18: 22, 47: 23, 55: 23, 56: 23, 54: 24, 64: 24,
    71: 25, 9: 26, 70: 27, 69: 28, 8: 29, 6: 30, 7: 30, 23: 30, 3: 31, 2: 32,
    4: 33, 29: 33, 68: 34, 5: 35, 27: 36, 28: 36, 1: 37, 10: 38, 0: 39, 26: 40, 67: 41
}

# 2. APLICAR AGRUPACIÓN HUMANA
# Usamos un try/except ligero por si alguna columna viene como Float desde BQ
df_raw_w6['id_agrupado'] = df_raw_w6['dropoff_polygon_id'].fillna(-1).astype(int).map(id_map).fillna(-1)

# 3. GENERAR LA "SALCHICHOTA" (Nombres Humanos)
name_foundry = df_raw_w6.groupby('id_agrupado')['dropoff_polygon_name'].unique().apply(lambda x: "__".join(sorted([str(i) for i in x if pd.notna(i)]))).to_dict()
df_raw_w6['grouped_polyname'] = df_raw_w6['id_agrupado'].map(name_foundry)

# 4. LA LÓGICA CORE (THE DECISION ENGINE)
# A. Logic for final_zone_id
conditions_id = [
    (df_raw_w6['id_agrupado'] >= 0),
    (df_raw_w6['dropoff_hdbscan_id'].fillna(-1).astype(int) > -1)
]
choices_id = [
    "P_" + df_raw_w6['id_agrupado'].astype(int).astype(str),
    "C_" + df_raw_w6['dropoff_hdbscan_id'].fillna(-1).astype(int).astype(str)
]
df_raw_w6['final_zone_id'] = np.select(conditions_id, choices_id, default="Unassigned")

# B. Logic for final_zone_name (Metadata for Streamlit)
conditions_name = [
    (df_raw_w6['id_agrupado'] >= 0),
    (df_raw_w6['dropoff_hdbscan_id'].fillna(-1).astype(int) > -1)
]
choices_name = [
    df_raw_w6['grouped_polyname'],
    df_raw_w6['dropoff_hdbscan_name']
]
df_raw_w6['final_zone_name'] = np.select(conditions_name, choices_name, default="Unassigned Area")

print("✅ Coalesce completado.")
print(f"-> Total Zonas Estratégicas Únicas en W6: {df_raw_w6['final_zone_id'].nunique()}")

# Verificación rápida de integridad
display(df_raw_w6[['dropoff_polygon_id', 'dropoff_hdbscan_id', 'final_zone_id', 'final_zone_name']].head(3))

⏳ Ejecutando Master Coalesce Geoespacial en W6...
✅ Coalesce completado.
-> Total Zonas Estratégicas Únicas en W6: 66


,dropoff_polygon_id,dropoff_hdbscan_id,final_zone_id,final_zone_name
0,-1,1,C_1,terminal_1_aicm
1,-1,-1,Unassigned,Unassigned Area
2,11,43,P_19,lomas_virreyes


In [4]:
# ==============================================================================
# CELDA 3: FEATURE ENGINEERING & SURVIVAL MATH (PURIFIED)
# ==============================================================================
# Purpose: Extract temporal features and apply logarithmic transformations.
# Constraint: ZERO product fusion (Independent ID 2 and ID 3).
# ==============================================================================
import numpy as np
import pandas as pd
from IPython.display import display

print("⏳ Ejecutando Ingeniería de Contexto y Matemática de Supervivencia...")

# --- 1. EXTRACCIÓN TEMPORAL ---
df_raw_w6['offer_timestamp'] = pd.to_datetime(df_raw_w6['offer_timestamp'])
df_raw_w6['hour_of_day'] = df_raw_w6['offer_timestamp'].dt.hour.astype(str)

if 'day_of_week' not in df_raw_w6.columns:
    df_raw_w6['day_of_week'] = df_raw_w6['offer_timestamp'].dt.day_name()
else:
    df_raw_w6['day_of_week'] = df_raw_w6['day_of_week'].astype(str)

# --- 2. PRODUCT CATEGORY (ZERO FUSION MODE) ---
# Convertimos a numérico solo para limpiar y luego a string para OHE
df_raw_w6['product_category_fk'] = pd.to_numeric(df_raw_w6['product_category_fk'], errors='coerce')

# 💎 LA CLAVE: No aplicamos .replace({3: 2}). 
# El 3 (Business) y el 2 (Comfort) se mantienen como entidades separadas.
df_raw_w6['product_category_fk'] = df_raw_w6['product_category_fk'].fillna("N/A").astype(str)

# --- 3. DEFINICIÓN DE LISTAS (PRAETORIAN GUARD L2) ---
praetorian_L2 = [
    'upfront_fare', 'est_trip_time_sec', 'is_multiple_destinations', 
    'session_progress_ratio', 'traffic_index_base_120', 'time_since_last_offer',
    'offer_density_10sec', 'consecutive_rejects', 'cycle_avg_dtp_km', 
    'cycle_std_dtp_km', 'cycle_ttp_dtp_ratio', 'dispatch_lead_time_sec', 
    'cycle_rolling_avg_spread', 'total_accumulated_deadhead_sec', 
    'cycle_cumulative_net_earnings', 'eph_operational_index', 
    'eph_complete_index_ML', 'home_vector_alignment_score', 'traffic_volatility_index_ml'
]

# --- 4. EXTRACCIÓN Y LIMPIEZA NUMÉRICA ---
X_num_w6 = df_raw_w6[praetorian_L2].copy()
for col in ['traffic_volatility_index_ml']:
    if col in X_num_w6.columns:
        X_num_w6[col] = pd.to_numeric(X_num_w6[col], errors='coerce')
X_num_w6 = X_num_w6.fillna(0)

# --- 5. TRANSFORMACIONES LOGARÍTMICAS ---
skewed_vars = [
    'upfront_fare', 'time_to_pickup_sec', 'est_trip_time_sec', 
    'traffic_index_base_120', 'total_accumulated_deadhead_sec', 
    'cycle_cumulative_net_earnings', 'traffic_volatility_index_ml'
]

for col in skewed_vars:
    if col in X_num_w6.columns:
        X_num_w6[f'log_{col}'] = np.log1p(X_num_w6[col].clip(lower=0))
        X_num_w6 = X_num_w6.drop(columns=[col])

print(f"✅ Ingeniería de variables completada (Zero Fusion).")
display(X_num_w6.head(2))

⏳ Ejecutando Ingeniería de Contexto y Matemática de Supervivencia...
✅ Ingeniería de variables completada (Zero Fusion).


,is_multiple_destinations,session_progress_ratio,time_since_last_offer,offer_density_10sec,consecutive_rejects,cycle_avg_dtp_km,cycle_std_dtp_km,cycle_ttp_dtp_ratio,dispatch_lead_time_sec,cycle_rolling_avg_spread,eph_operational_index,eph_complete_index_ML,home_vector_alignment_score,log_upfront_fare,log_est_trip_time_sec,log_traffic_index_base_120,log_total_accumulated_deadhead_sec,log_cycle_cumulative_net_earnings,log_traffic_volatility_index_ml
0,0,0.001726,19.0,1,1,2.050000,0.450000,177.750000,0.0,0.0,1.496625,0.0,-0.919873,4.793557,7.039660,0.482991,2.995732,0.0,0.0
1,0,0.004906,35.0,1,2,1.833333,0.478423,189.928571,0.0,0.0,1.301053,0.0,0.609556,4.423648,6.734592,0.747214,4.007333,0.0,0.0


In [5]:
# ==============================================================================
# CELDA 4: THE ALIGNMENT FORGE (GCS + CAPA 1) - PURIFIED
# ==============================================================================
# Purpose: Align the Week 6 matrix to the exact schema expected by the Portero.
# ==============================================================================
from google.cloud import storage
import joblib
import re
import os
import pandas as pd
from IPython.display import display

print("⏳ Extrayendo la Trinidad de Modelos desde GCS...")

# 1. CARGA DEL MODELO
BUCKET_NAME = 'pienza-streamlit'
BLOB_PATH = '260419_XGB_models.joblib' 
LOCAL_TMP_PATH = '/tmp/260419_XGB_models.joblib'

try:
    storage_client = storage.Client()
    bucket = storage_client.bucket(BUCKET_NAME)
    blob = bucket.blob(BLOB_PATH)
    blob.download_to_filename(LOCAL_TMP_PATH)
    
    arena_assets = joblib.load(LOCAL_TMP_PATH)
    model_champion_L1 = arena_assets['model_L1']
    features_l1_oficial = model_champion_L1.feature_names_in_
    print(f"✅ Esquema del Portero extraído ({len(features_l1_oficial)} features).")
except Exception as e:
    print(f"🔴 ERROR FATAL: {e}")
    features_l1_oficial = []

if len(features_l1_oficial) > 0:
    print("\n⏳ Alineando 780 viajes para Capa 1 (Zero-Fusion Mode)...")

    # 2. DEFINICIÓN DE FEATURES CAPA 1
    numeric_L1 = [
        'upfront_fare', 'time_to_pickup_sec', 'est_trip_time_sec',
        'traffic_index_base_120', 'time_since_last_offer',
        'eph_operational_index', 'historical_rolling_avg_traffic_index',
        'traffic_volatility_index_ml'
    ]
    categorical_L1 = [
        'product_category_fk', 'hour_of_day', 'day_of_week', 'final_zone_id'
    ]

    # 3. PROCESAMIENTO NUMÉRICO L1
    X_num_L1_w6 = df_raw_w6[numeric_L1].copy().fillna(0)
    for col in ['historical_rolling_avg_traffic_index', 'traffic_volatility_index_ml']:
        X_num_L1_w6[col] = pd.to_numeric(X_num_L1_w6[col], errors='coerce').fillna(0)

    # 4. PROCESAMIENTO CATEGÓRICO L1 (SIN FUSIÓN)
    X_cat_L1_w6 = df_raw_w6[categorical_L1].copy()
    X_cat_L1_w6['product_category_fk'] = X_cat_L1_w6['product_category_fk'].astype(str)
    X_cat_L1_w6 = X_cat_L1_w6.fillna("N/A")

    # 5. ONE-HOT ENCODING (Aquí nacerá la columna _3 de forma natural)
    X_cat_encoded_L1_w6 = pd.get_dummies(X_cat_L1_w6, drop_first=True)

    # 6. FUSIÓN Y LIMPIEZA REGEX
    X_L1_raw_w6 = pd.concat([X_num_L1_w6, X_cat_encoded_L1_w6], axis=1)
    regex = re.compile(r"\[|\]|<", re.IGNORECASE)
    X_L1_raw_w6.columns = [
        regex.sub("_", col) if any(x in str(col) for x in set(('[', ']', '<'))) else col 
        for col in X_L1_raw_w6.columns
    ]

    # 7. ALINEACIÓN ESTRICTA (EL SECRETO DEL 0.0)
    X_L1_final = X_L1_raw_w6.reindex(columns=features_l1_oficial, fill_value=0)

    print(f"✅ MATRIZ X_L1_final LISTA ({X_L1_final.shape[1]} columnas alineadas).")
    display(X_L1_final.head(2))

⏳ Extrayendo la Trinidad de Modelos desde GCS...
✅ Esquema del Portero extraído (103 features).

⏳ Alineando 780 viajes para Capa 1 (Zero-Fusion Mode)...
✅ MATRIZ X_L1_final LISTA (103 columnas alineadas).


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,upfront_fare,time_to_pickup_sec,est_trip_time_sec,traffic_index_base_120,time_since_last_offer,eph_operational_index,historical_rolling_avg_traffic_index,traffic_volatility_index_ml,product_category_fk_2,product_category_fk_3,...,final_zone_id_P_39,final_zone_id_P_4,final_zone_id_P_40,final_zone_id_P_41,final_zone_id_P_5,final_zone_id_P_6,final_zone_id_P_7,final_zone_id_P_8,final_zone_id_P_9,final_zone_id_Unassigned
0,119.73,300.0,1140.0,0.620915,19.0,1.496625,0.0,0.0,False,True,...,False,False,False,False,False,False,False,False,False,False
1,82.40,300.0,840.0,1.111111,35.0,1.301053,0.0,0.0,True,False,...,False,False,False,False,False,False,False,False,False,True


In [6]:
# ==============================================================================
# CELDA 5: TEST DE PARIDAD TOTAL (ZERO TOLERANCE)
# ==============================================================================
# Purpose: Final audit to achieve 0.0 difference between BQ and Colab.
# ==============================================================================
from google.cloud import storage
import pandas as pd
import numpy as np

print("⚖️ Iniciando Peritaje de Paridad Total (Zero Tolerance)...")

# --- 1. CONFIGURACIÓN GCS ---
BUCKET_NAME = 'pienza-streamlit'
BLOB_PATH_PARQUET = '260419_paridad_780_viajes_limpios.parquet'
LOCAL_PARQUET_PATH = '/tmp/paridad_final.parquet'

# --- 2. DESCARGA Y CARGA ---
try:
    storage_client = storage.Client()
    bucket = storage_client.bucket(BUCKET_NAME)
    blob = bucket.blob(BLOB_PATH_PARQUET)
    blob.download_to_filename(LOCAL_PARQUET_PATH)
    
    # Cargamos el Master de Colab
    x_l1_colab = pd.read_parquet(LOCAL_PARQUET_PATH)
    print(f"✅ Golden Master V4 cargado: {x_l1_colab.shape}")
except Exception as e:
    print(f"🔴 ERROR: Fallo al obtener el parquet de GCS: {e}")
    x_l1_colab = None

if x_l1_colab is not None:
    # 3. ALINEACIÓN Y CÁLCULO DE DIFERENCIAS
    # Aseguramos que el orden de las columnas sea idéntico
    common_cols = x_l1_colab.columns.tolist()
    
    # Comparamos sumatorias con redondeo para evitar ruidos de precisión decimal
    sum_colab = x_l1_colab[common_cols].sum().round(4)
    sum_bq    = X_L1_final[common_cols].sum().round(4)
    
    diff_vector = (sum_colab - sum_bq).abs()
    max_total_diff = diff_vector.max()
    
    # 4. VERDICTO ---
    print("\n🔬 RESULTADO DEL PERITAJE FINAL:")
    if max_total_diff == 0:
        print("💎 PARIDAD ABSOLUTA LOGRADA (DIFF = 0.0000)")
        print("El pipeline de BigQuery es un espejo perfecto de Colab.")
        print("🚀 ESTADO: INFRAESTRUCTURA CERTIFICADA. PROCEDE AL DASHBOARD.")
    elif max_total_diff < 0.0001:
        print(f"🟢 PARIDAD VIRTUAL LOGRADA (DIFF ≈ {max_total_diff})")
        print("Diferencia despreciable (ruido de flotantes).")
        print("🚀 ESTADO: CERTIFICADO.")
    else:
        print(f"🔴 DISCREPANCIA PERSISTENTE. Max Diff: {max_total_diff}")
        top_errors = diff_vector[diff_vector > 0].sort_values(ascending=False)
        print("\n⚠️ Variables que aún no cuadran:")
        print(top_errors.head(10))

⚖️ Iniciando Peritaje de Paridad Total (Zero Tolerance)...
✅ Golden Master V4 cargado: (780, 103)

🔬 RESULTADO DEL PERITAJE FINAL:
💎 PARIDAD ABSOLUTA LOGRADA (DIFF = 0.0000)
El pipeline de BigQuery es un espejo perfecto de Colab.
🚀 ESTADO: INFRAESTRUCTURA CERTIFICADA. PROCEDE AL DASHBOARD.


In [8]:
# ==============================================================================
# AUDITORÍA DE CAMPOS: v_ML_Supervised (Gold Layer)
# Purpose: Identificar nombres exactos y tipos de datos para evitar KeyErrors.
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

# 1. Referencia a la tabla
table_id = f"{PROJECT_ID}.{DATASET_CORE}.v_ML_Supervised"

print(f"🕵️‍♂️ Escaneando metadatos de: {table_id}...")

try:
    # 2. Obtener el objeto tabla desde BigQuery
    table = client.get_table(table_id)
    
    # 3. Extraer esquema
    schema_info = []
    for field in table.schema:
        schema_info.append({
            "Field Name": field.name,
            "Type": field.field_type,
            "Mode": field.mode,
            "Description": field.description if field.description else ""
        })

    # 4. Visualización elegante con Pandas
    df_schema = pd.DataFrame(schema_info)
    
    display(Markdown(f"### 📋 Inventario de Columnas (`v_ML_Supervised`)"))
    display(Markdown(f"Total de campos detectados: `{len(df_schema)}`"))
    
    # Aplicamos estilo Opus Lab para que se vea profesional
    display(df_schema.style.set_properties(**{
        'background-color': '#FAFAFA',
        'color': '#121212',
        'border-color': '#DDDDDD'
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#440154'), ('color', 'white')]}
    ]))

    # 5. Lista rápida para copiar y pegar (en caso de error)
    print("\n📝 Lista de nombres (copy-paste ready):")
    print(df_schema["Field Name"].tolist())

except Exception as e:
    print(f"❌ Error al consultar BigQuery: {e}")

🕵️‍♂️ Escaneando metadatos de: 645009831643.pienza_mini.v_ML_Supervised...


### 📋 Inventario de Columnas (`v_ML_Supervised`)

Total de campos detectados: `105`

,Field Name,Type,Mode,Description
0,offer_id,STRING,NULLABLE,
1,session_fk,STRING,NULLABLE,
2,ocr_fk,STRING,NULLABLE,
3,image_content_hash,STRING,NULLABLE,
4,offer_timestamp,STRING,NULLABLE,
5,hour_of_day,INTEGER,NULLABLE,
6,upfront_fare,FLOAT,NULLABLE,
7,time_to_pickup_sec,FLOAT,NULLABLE,
8,dist_to_pickup_km,FLOAT,NULLABLE,
9,est_trip_time_sec,FLOAT,NULLABLE,



📝 Lista de nombres (copy-paste ready):
['offer_id', 'session_fk', 'ocr_fk', 'image_content_hash', 'offer_timestamp', 'hour_of_day', 'upfront_fare', 'time_to_pickup_sec', 'dist_to_pickup_km', 'est_trip_time_sec', 'est_trip_dist_km', 'pickup_address', 'dropoff_address', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon', 'is_surge', 'surge_amount', 'is_turbo_plus', 'turbo_plus_amount', 'is_reservation', 'reservation_amount', 'is_priority', 'priority_amount', 'is_exclusive', 'is_vip', 'is_identity_verified', 'is_long_trip', 'is_multiple_destinations', 'is_teens', 'rider_star_rating', 'rider_trip_count', 'time_in_session_sec', 'session_progress_ratio', 'inferred_agent_lat', 'inferred_agent_lon', 'inferred_agent_bearing', 'inferred_agent_speed_mps', 'is_imputed', 'special_note_raw', 'comment_1', 'comment_2', 'product_category_fk', 'offer_action_fk', 'reason_primary_fk', 'post_offer_status_fk', 'driver_state_at_request_fk', 'outcome_fk', 'interpolation_quality_fk', 'record_status_fk',